# Lab 2: Spark Fundamentals & the Medallion Architecture

**DAT535 - Data Engineering**

This lab is the foundation for everything that follows in **Lab 3 (Advanced Spark & Production Patterns)**.
Both labs share **one single e-commerce clickstream dataset**, generated once in this notebook and
persisted to disk so that Lab 3 can load exactly the same data.

## Learning Objectives

By the end of this lab you will be able to:

- Start and configure a `SparkSession`
- Understand the Spark driver / executor / task / partition model
- Create Spark **DataFrames** and **RDDs**, and convert between **RDD ↔ DataFrame ↔ SQL**
- Use the DataFrame API: `select`, `withColumn`, casting, filtering, sorting, aggregating, joining basics
- Apply **MapReduce** patterns (`map`, `flatMap`, `reduceByKey`) directly with RDDs
- Read and write data in **Parquet, CSV and JSON**, including partitioned writes
- Build a **Bronze → Silver → Gold (Medallion) pipeline** with data-quality quarantining

## The Shared Dataset

Throughout **both** labs we use a single, consistently-shaped e-commerce clickstream dataset that
simulates raw JSON events arriving from a web/mobile application (page views, searches, cart actions,
purchases, logins). A small percentage of records are **intentionally malformed** so we have something
real to clean in the Silver layer.

In [2]:
# Setup and Imports
import os
import json
import time
from datetime import datetime, timedelta
import random
from pyspark.sql.functions import expr

import findspark
findspark.init()

from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

print("Libraries imported successfully")


Libraries imported successfully


In [3]:
# Create the SparkSession - the entry point for all Spark functionality
spark = SparkSession.builder \
    .appName("DAT535-Lab2-SparkFundamentals") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/02 11:38:27 WARN Utils: Your hostname, Martins-MacBook-Air-4.local, resolves to a loopback address: 127.0.0.1; using 10.14.132.120 instead (on interface en0)
26/10/02 11:38:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/02 11:38:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.0.4
Application ID: local-1790933910139
Spark UI: http://10.14.132.120:4040


### Understanding Spark Architecture

```
+---------------------------------------------------------------+
|                      Driver Program                           |
|  +-----------------------------------------------------+      |
|  |              SparkSession / SparkContext             |      |
|  +-----------------------------------------------------+      |
+---------------------------------------------------------------+
                              |
              +---------------+---------------+
              v               v               v
        +----------+   +----------+   +----------+
        | Executor |   | Executor |   | Executor |
        |  Task 1  |   |  Task 2  |   |  Task 3  |
        |  Task 4  |   |  Task 5  |   |  Task 6  |
        +----------+   +----------+   +----------+
```

**Key concepts:** the **driver** coordinates the application, **executors** are worker processes,
**tasks** are units of work sent to executors, and **partitions** are the chunks of data processed
in parallel.


## Part 1: Generating the Shared E-Commerce Dataset

In [4]:
def generate_ecommerce_data(num_events=6000, num_users=200, num_products=100, seed=42):
    """Generate the canonical e-commerce clickstream dataset used across Lab 2 and Lab 3.

    A small fraction of records are intentionally malformed (bad user_id / bad timestamp)
    so that the Silver layer quality gates later in this notebook have something to catch.
    """
    random.seed(seed)

    event_types = ['page_view', 'search', 'add_to_cart', 'remove_from_cart',
                   'purchase', 'login', 'logout', 'wishlist_add']
    devices = ['mobile', 'desktop', 'tablet']
    categories = ['Electronics', 'Clothing', 'Books', 'Home', 'Sports', 'Beauty']
    countries = ['US', 'UK', 'DE', 'FR', 'CA', 'AU', 'JP', 'IN']

    events = []
    base_time = datetime(2024, 1, 1, 0, 0, 0)

    for i in range(num_events):
        timestamp = base_time + timedelta(
            days=random.randint(0, 29),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )

        event_type = random.choice(event_types)
        user_id = random.randint(1, num_users)

        event = {
            'event_id': f'evt_{i+1:06d}',
            'timestamp': timestamp.isoformat(),
            'user_id': user_id,
            'event_type': event_type,
            'device': random.choice(devices),
            'country': random.choice(countries),
            'session_id': f'sess_{user_id}_{random.randint(1, 5):03d}',
            'product_id': None,
            'category': None,
            'price': None,
            'quantity': None,
            'total_amount': None,
            'search_query': None,
        }

        if event_type in ['page_view', 'add_to_cart', 'purchase', 'wishlist_add']:
            event['product_id'] = f'prod_{random.randint(1, num_products):04d}'
            event['category'] = random.choice(categories)
            event['price'] = round(random.uniform(9.99, 499.99), 2)

        if event_type == 'purchase':
            event['quantity'] = random.randint(1, 5)
            event['total_amount'] = round(event['price'] * event['quantity'], 2)

        if event_type == 'search':
            event['search_query'] = random.choice(
                ['laptop', 'shoes', 'phone', 'book', 'jacket', 'headphones'])

        # Inject ~3% bad data so the Medallion pipeline has real quality issues to catch
        if random.random() < 0.03:
            if random.random() < 0.5:
                event['user_id'] = 'invalid'
            else:
                event['timestamp'] = 'not-a-date'

        events.append(event)

    return events

# Generate the data - this exact dataset (via the fixed seed) is regenerated in Lab 3 as well
raw_events = generate_ecommerce_data(num_events=6000, num_users=200, num_products=100)
raw_json_events = [json.dumps(e) for e in raw_events]

print(f"Generated {len(raw_events)} events")
print("\nSample event:")
print(json.dumps(raw_events[0], indent=2))


Generated 6000 events

Sample event:
{
  "event_id": "evt_000001",
  "timestamp": "2024-01-21T03:01:47",
  "user_id": 63,
  "event_type": "purchase",
  "device": "mobile",
  "country": "DE",
  "session_id": "sess_63_001",
  "product_id": "prod_0087",
  "category": "Beauty",
  "price": 447.16,
  "quantity": 1,
  "total_amount": 447.16,
  "search_query": null
}


## Part 2: Creating & Converting DataFrames

One of the most important Spark skills is knowing how to move data between the different
representations: **RDD**, **DataFrame**, and **SQL view** - all native to Spark, with no
extra dependencies required.

In [5]:
# Conversion 1: Python list[dict] -> DataFrame (schema inference)
events_df = spark.createDataFrame(raw_events)
print("=== Inferred Schema ===")
events_df.printSchema()
print(f"Rows: {events_df.count()}, Columns: {len(events_df.columns)}")


=== Inferred Schema ===
root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- search_query: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- user_id: string (nullable = true)



Rows: 6000, Columns: 13


In [6]:
# Conversion 2: Raw JSON strings -> RDD -> DataFrame (spark.read.json on an RDD of strings)
json_rdd = spark.sparkContext.parallelize(raw_json_events)
events_from_json_df = spark.read.json(json_rdd)
print("=== Schema when reading directly from JSON strings ===")
events_from_json_df.printSchema()


=== Schema when reading directly from JSON strings ===
root
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- search_query: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
# Conversion 3: DataFrame -> RDD of Row objects, and back to a DataFrame
events_rdd = events_df.rdd
print("First Row object:", events_rdd.first())

# RDD[Row] -> DataFrame again (toDF)
roundtrip_df = events_rdd.toDF()
print(f"Round-tripped rows: {roundtrip_df.count()}")


First Row object: Row(category='Beauty', country='DE', device='mobile', event_id='evt_000001', event_type='purchase', price=447.16, product_id='prod_0087', quantity=1, search_query=None, session_id='sess_63_001', timestamp='2024-01-21T03:01:47', total_amount=447.16, user_id='63')
Round-tripped rows: 6000


In [8]:
# Conversion 4: Explicit schema definition (recommended for production - avoids costly inference)
explicit_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("timestamp", StringType(), True),
    StructField("user_id", StringType(), True),   # kept as string: some rows contain 'invalid'
    StructField("event_type", StringType(), True),
    StructField("device", StringType(), True),
    StructField("country", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("search_query", StringType(), True),
])

events_typed_df = spark.createDataFrame(raw_events, schema=explicit_schema)
events_typed_df.printSchema()


root
 |-- event_id: string (nullable = false)
 |-- timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- search_query: string (nullable = true)



In [9]:
# Conversion 5: DataFrame <-> SQL views
events_typed_df.createOrReplaceTempView("events")

sql_result = spark.sql("""
    SELECT event_type, COUNT(*) AS event_count
    FROM events
    GROUP BY event_type
    ORDER BY event_count DESC
""")

print("=== SQL query result (still a DataFrame!) ===")
sql_result.show()

# The result of spark.sql() is a normal DataFrame - it can be filtered/joined/etc. like any other
print(type(sql_result))


=== SQL query result (still a DataFrame!) ===
+----------------+-----------+
|      event_type|event_count|
+----------------+-----------+
|remove_from_cart|        787|
|          logout|        787|
|           login|        771|
|       page_view|        756|
|    wishlist_add|        735|
|     add_to_cart|        730|
|          search|        729|
|        purchase|        705|
+----------------+-----------+

<class 'pyspark.sql.classic.dataframe.DataFrame'>


## Part 3: Column Operations, Type Casting & Null Handling

In [10]:
# select(): choose specific columns
print("=== Selecting Specific Columns ===")
events_df.select("event_id", "user_id", "event_type", "timestamp").show(5)

# col() + alias(): recommended for anything beyond trivial column selection
events_df.select(
    col("event_id"),
    col("user_id"),
    col("event_type").alias("action"),
    col("device")
).show(5)


=== Selecting Specific Columns ===
+----------+-------+----------------+-------------------+
|  event_id|user_id|      event_type|          timestamp|
+----------+-------+----------------+-------------------+
|evt_000001|     63|        purchase|2024-01-21T03:01:47|
|evt_000002|    130|remove_from_cart|2024-01-02T00:05:13|
|evt_000003|    115|remove_from_cart|2024-01-21T22:34:26|
|evt_000004|     40|        purchase|2024-01-06T22:27:21|
|evt_000005|     97|          search|2024-01-26T01:46:29|
+----------+-------+----------------+-------------------+
only showing top 5 rows
+----------+-------+----------------+------+
|  event_id|user_id|          action|device|
+----------+-------+----------------+------+
|evt_000001|     63|        purchase|mobile|
|evt_000002|    130|remove_from_cart|tablet|
|evt_000003|    115|remove_from_cart|tablet|
|evt_000004|     40|        purchase|mobile|
|evt_000005|     97|          search|mobile|
+----------+-------+----------------+------+
only showing t

In [11]:
# withColumn(): create/replace columns, and cast() to convert types
# Chain of conversions: string -> timestamp -> date / hour / day-of-week
enhanced_df = events_df \
    .withColumn("user_id_int", expr("try_cast(user_id AS INT)")) \
    .withColumn("timestamp_parsed", expr("try_cast(timestamp AS TIMESTAMP)")) \
    .withColumn("event_date", to_date(col("timestamp_parsed"))) \
    .withColumn("event_hour", hour(col("timestamp_parsed"))) \
    .withColumn("day_of_week", dayofweek(col("event_date"))) \
    .withColumn("is_purchase", when(col("event_type") == "purchase", True).otherwise(False)) \
    .withColumn("device_upper", upper(col("device")))

enhanced_df.select(
    "event_id", "user_id", "user_id_int", "timestamp",
    "timestamp_parsed", "event_date", "event_hour", "is_purchase"
).show(8, truncate=False)

print("Rows where user_id failed to cast to int (the 'invalid' bad-data rows):")
enhanced_df.filter(col("user_id_int").isNull()).select("event_id", "user_id").show(5)


+----------+-------+-----------+-------------------+-------------------+----------+----------+-----------+
|event_id  |user_id|user_id_int|timestamp          |timestamp_parsed   |event_date|event_hour|is_purchase|
+----------+-------+-----------+-------------------+-------------------+----------+----------+-----------+
|evt_000001|63     |63         |2024-01-21T03:01:47|2024-01-21 03:01:47|2024-01-21|3         |true       |
|evt_000002|130    |130        |2024-01-02T00:05:13|2024-01-02 00:05:13|2024-01-02|0         |false      |
|evt_000003|115    |115        |2024-01-21T22:34:26|2024-01-21 22:34:26|2024-01-21|22        |false      |
|evt_000004|40     |40         |2024-01-06T22:27:21|2024-01-06 22:27:21|2024-01-06|22        |true       |
|evt_000005|97     |97         |2024-01-26T01:46:29|2024-01-26 01:46:29|2024-01-26|1         |false      |
|evt_000006|198    |198        |2024-01-23T02:02:42|2024-01-23 02:02:42|2024-01-23|2         |false      |
|evt_000007|42     |42         |2024-

In [12]:
# Handling nulls: isNull/isNotNull, na.fill, na.drop, coalesce
print("Events missing a price:", events_df.filter(col("price").isNull()).count())
print("Events with a price:", events_df.filter(col("price").isNotNull()).count())

filled_df = events_df.na.fill({"price": 0.0, "category": "Unknown"})
dropped_df = events_df.na.drop(subset=["event_id"])
coalesced_df = events_df.withColumn("category_or_default", coalesce(col("category"), lit("Unknown")))

print("na.fill / na.drop / coalesce all applied successfully")
coalesced_df.select("event_id", "category", "category_or_default").show(5)


Events missing a price: 3074
Events with a price: 2926
na.fill / na.drop / coalesce all applied successfully
+----------+--------+-------------------+
|  event_id|category|category_or_default|
+----------+--------+-------------------+
|evt_000001|  Beauty|             Beauty|
|evt_000002|    NULL|            Unknown|
|evt_000003|    NULL|            Unknown|
|evt_000004|    Home|               Home|
|evt_000005|    NULL|            Unknown|
+----------+--------+-------------------+
only showing top 5 rows


In [13]:
# distinct / dropDuplicates and union
distinct_devices = events_df.select("device").distinct()
print("Distinct devices:")
distinct_devices.show()

deduped = events_df.dropDuplicates(["session_id"])
print(f"Rows before dedup: {events_df.count()}, unique sessions: {deduped.count()}")

# union(): stack two DataFrames with the same schema
mobile_events = events_df.filter(col("device") == "mobile")
desktop_events = events_df.filter(col("device") == "desktop")
combined = mobile_events.union(desktop_events)
print(f"mobile ({mobile_events.count()}) + desktop ({desktop_events.count()}) = {combined.count()}")


Distinct devices:
+-------+
| device|
+-------+
| tablet|
| mobile|
|desktop|
+-------+

Rows before dedup: 6000, unique sessions: 997
mobile (1936) + desktop (2029) = 3965


## Part 4: Filtering Data

In [14]:
print("=== Filtering Examples ===")

purchases_df = events_df.filter(col("event_type") == "purchase")
print(f"Total purchases: {purchases_df.count()}")

mobile_purchases = events_df.filter(
    (col("event_type") == "purchase") & (col("device") == "mobile")
)
print(f"Mobile purchases: {mobile_purchases.count()}")

engagement_events = events_df.filter(
    (col("event_type") == "add_to_cart") | (col("event_type") == "wishlist_add")
)
print(f"Engagement events: {engagement_events.count()}")

transaction_events = events_df.filter(
    col("event_type").isin(["purchase", "add_to_cart", "remove_from_cart"])
)
print(f"Transaction events: {transaction_events.count()}")

high_value = events_df.filter(
    (col("total_amount").isNotNull()) & (col("total_amount") > 100)
)
print(f"High-value purchases (>$100): {high_value.count()}")
high_value.select("event_id", "user_id", "product_id", "total_amount").show(5)


=== Filtering Examples ===
Total purchases: 705
Mobile purchases: 235
Engagement events: 1465
Transaction events: 2222
High-value purchases (>$100): 660
+----------+-------+----------+------------+
|  event_id|user_id|product_id|total_amount|
+----------+-------+----------+------------+
|evt_000001|     63| prod_0087|      447.16|
|evt_000004|     40| prod_0012|      172.14|
|evt_000019|    103| prod_0067|      138.56|
|evt_000047|    130| prod_0052|      309.72|
|evt_000055|    197| prod_0036|      143.88|
+----------+-------+----------+------------+
only showing top 5 rows


## Part 5: Sorting Data

In [15]:
print("Earliest events:")
events_df.orderBy("timestamp").select("event_id", "timestamp", "event_type").show(5)

print("Latest events:")
events_df.orderBy(desc("timestamp")).select("event_id", "timestamp", "event_type").show(5)

print("Sorted by user_id (asc) then timestamp (desc):")
events_df.orderBy(asc("user_id"), desc("timestamp")).select("user_id", "timestamp", "event_type").show(10)


Earliest events:
+----------+-------------------+----------------+
|  event_id|          timestamp|      event_type|
+----------+-------------------+----------------+
|evt_003339|2024-01-01T00:14:00|          search|
|evt_004375|2024-01-01T00:20:58|remove_from_cart|
|evt_003871|2024-01-01T00:23:55|remove_from_cart|
|evt_002704|2024-01-01T00:39:45|           login|
|evt_003053|2024-01-01T00:40:04|        purchase|
+----------+-------------------+----------------+
only showing top 5 rows
Latest events:
+----------+----------+----------------+
|  event_id| timestamp|      event_type|
+----------+----------+----------------+
|evt_003472|not-a-date|    wishlist_add|
|evt_002083|not-a-date|remove_from_cart|
|evt_000244|not-a-date|    wishlist_add|
|evt_002198|not-a-date|     add_to_cart|
|evt_004824|not-a-date|    wishlist_add|
+----------+----------+----------------+
only showing top 5 rows
Sorted by user_id (asc) then timestamp (desc):
+-------+-------------------+----------------+
|user_i

## Part 6: Aggregations & Grouping

In [16]:
print("=== Global Aggregations ===")
events_df.agg(
    count("*").alias("total_events"),
    countDistinct("user_id").alias("unique_users"),
    countDistinct("session_id").alias("unique_sessions"),
    spark_sum("total_amount").alias("total_revenue"),
    avg("total_amount").alias("avg_order_value"),
    spark_max("total_amount").alias("max_order_value")
).show(truncate=False)


=== Global Aggregations ===


+------------+------------+---------------+-------------+-----------------+---------------+
|total_events|unique_users|unique_sessions|total_revenue|avg_order_value  |max_order_value|
+------------+------------+---------------+-------------+-----------------+---------------+
|6000        |201         |997            |572554.14    |812.1335319148936|2499.75        |
+------------+------------+---------------+-------------+-----------------+---------------+



In [17]:
print("=== Event Type Distribution ===")
events_df.groupBy("event_type").agg(
    count("*").alias("event_count"),
    countDistinct("user_id").alias("unique_users")
).orderBy(desc("event_count")).show()


=== Event Type Distribution ===
+----------------+-----------+------------+
|      event_type|event_count|unique_users|
+----------------+-----------+------------+
|remove_from_cart|        787|         198|
|          logout|        787|         195|
|           login|        771|         197|
|       page_view|        756|         193|
|    wishlist_add|        735|         197|
|     add_to_cart|        730|         195|
|          search|        729|         195|
|        purchase|        705|         192|
+----------------+-----------+------------+



In [18]:
print("=== Sales by Device and Country ===")
purchases = events_df.filter(col("event_type") == "purchase")

purchases.groupBy("device", "country").agg(
    count("*").alias("num_purchases"),
    spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
    spark_round(avg("total_amount"), 2).alias("avg_order_value"),
    countDistinct("user_id").alias("unique_buyers")
).orderBy(desc("total_revenue")).show(15)


=== Sales by Device and Country ===
+-------+-------+-------------+-------------+---------------+-------------+
| device|country|num_purchases|total_revenue|avg_order_value|unique_buyers|
+-------+-------+-------------+-------------+---------------+-------------+
| tablet|     UK|           39|     33071.85|          848.0|           36|
| mobile|     FR|           32|     29927.17|         935.22|           30|
| mobile|     CA|           33|     29722.29|         900.68|           29|
|desktop|     UK|           32|     29273.87|         914.81|           31|
|desktop|     CA|           33|      28242.0|         855.82|           30|
|desktop|     JP|           32|     28081.68|         877.55|           30|
|desktop|     FR|           28|     28006.29|        1000.22|           27|
| mobile|     IN|           32|     27672.29|         864.76|           31|
| tablet|     IN|           33|     27632.15|         837.34|           30|
|desktop|     IN|           34|      26099.0|       

In [19]:
print("=== User Engagement (exercise) ===")
user_engagement = events_df.groupBy("user_id").agg(
    count("*").alias("total_events"),
    countDistinct("session_id").alias("num_sessions"),
    countDistinct("event_type").alias("event_variety"),
    collect_list("event_type").alias("events_list")
).orderBy(desc("total_events"))

user_engagement.show(10, truncate=False)


=== User Engagement (exercise) ===
+-------+------------+------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Part 7: Reading & Writing Data (Parquet, CSV, JSON, partitioned)

In [20]:
SHARED_DIR = os.path.expanduser("~/spark-lab-data/shared")
LAB2_DIR = os.path.expanduser("~/spark-lab-data/lab2")
os.makedirs(SHARED_DIR, exist_ok=True)
os.makedirs(LAB2_DIR, exist_ok=True)
print(f"Shared dataset directory (read by Lab 3 too): {SHARED_DIR}")
print(f"Lab 2 scratch directory: {LAB2_DIR}")


Shared dataset directory (read by Lab 3 too): /Users/martin/spark-lab-data/shared
Lab 2 scratch directory: /Users/martin/spark-lab-data/lab2


In [21]:
# Parquet - the recommended columnar format for big data (compressed, typed, splittable)
parquet_path = f"{LAB2_DIR}/events.parquet"
events_df.write.mode("overwrite").parquet(parquet_path)
print(f"Written to: {parquet_path}")
print(f"Read back: {spark.read.parquet(parquet_path).count()} rows")


Written to: /Users/martin/spark-lab-data/lab2/events.parquet
Read back: 6000 rows


In [22]:
# CSV - human readable, but no schema/types are preserved unless you infer or supply one
csv_path = f"{LAB2_DIR}/events.csv"
events_df.write.mode("overwrite").option("header", "true").csv(csv_path)
print(f"Written to: {csv_path}")

csv_df = spark.read.option("header", "true").option("inferSchema", "true").csv(csv_path)
print(f"Read back: {csv_df.count()} rows")


Written to: /Users/martin/spark-lab-data/lab2/events.csv
Read back: 6000 rows


In [23]:
# JSON - convenient for semi-structured data, one JSON object per line
json_path = f"{LAB2_DIR}/events.json"
events_df.write.mode("overwrite").json(json_path)
print(f"Written to: {json_path}")
print(f"Read back: {spark.read.json(json_path).count()} rows")


Written to: /Users/martin/spark-lab-data/lab2/events.json
Read back: 6000 rows


In [24]:
# Partitioned writes - critical for large datasets and predicate pushdown
partitioned_path = f"{LAB2_DIR}/events_partitioned.parquet"
partitioned_df = enhanced_df.select(*events_df.columns, "event_date")

partitioned_df.write.mode("overwrite").partitionBy("event_date").parquet(partitioned_path)
print(f"Written partitioned data to: {partitioned_path}")

print("Reading with a partition filter (partition pruning avoids scanning every file):")
sample_date = partitioned_df.select("event_date").first()["event_date"]
filtered_read = spark.read.parquet(partitioned_path).filter(col("event_date") == sample_date)
print(f"Events on {sample_date}: {filtered_read.count()}")


Written partitioned data to: /Users/martin/spark-lab-data/lab2/events_partitioned.parquet
Reading with a partition filter (partition pruning avoids scanning every file):
Events on 2024-01-21: 218


## Part 8: RDD Transformations & MapReduce Patterns

Every DataFrame operation ultimately compiles down to RDD operations under the hood. Understanding
the classic **Map -> Shuffle -> Reduce** pattern is essential for reasoning about performance and for
writing custom transformations that the DataFrame API cannot express.


In [25]:
print("=== RDD Transformations ===")

# map(): 1-to-1 transformation
user_ids_rdd = events_rdd.map(lambda row: row["user_id"])
print("First 5 user_ids (map):", user_ids_rdd.take(5))

# flatMap(): 1-to-many transformation (flattens the results)
words_rdd = spark.sparkContext.parallelize(["spark is fast", "rdds are low level"]) \
    .flatMap(lambda line: line.split(" "))
print("flatMap result:", words_rdd.collect())

# filter() on an RDD
purchase_rows_rdd = events_rdd.filter(lambda row: row["event_type"] == "purchase")
print("Purchases via RDD filter:", purchase_rows_rdd.count())

# distinct()
print("Distinct devices via RDD:", events_rdd.map(lambda r: r["device"]).distinct().collect())


=== RDD Transformations ===
First 5 user_ids (map): ['63', '130', '115', '40', '97']
flatMap result: ['spark', 'is', 'fast', 'rdds', 'are', 'low', 'level']
Purchases via RDD filter: 705
Distinct devices via RDD: ['mobile', 'tablet', 'desktop']


In [26]:
print("=== Classic MapReduce: Word Count ===")

sample_text = [
    "Apache Spark is a unified analytics engine for big data processing",
    "Spark provides high-level APIs in Java Scala Python and R",
    "Spark runs on Hadoop YARN Kubernetes and standalone clusters",
    "The Spark ecosystem includes Spark SQL for structured data processing",
    "Spark Streaming enables scalable and fault-tolerant stream processing",
]

text_rdd = spark.sparkContext.parallelize(sample_text)

# MAP phase: split lines into (word, 1) pairs
word_pairs_rdd = text_rdd.flatMap(lambda line: line.lower().split()).map(lambda w: (w, 1))

# REDUCE phase: sum counts per key - reduceByKey shuffles data by key then combines
word_counts_rdd = word_pairs_rdd.reduceByKey(lambda a, b: a + b)

print("Top 10 words (RDD MapReduce):")
for word, cnt in word_counts_rdd.sortBy(lambda x: -x[1]).take(10):
    print(f"  {word}: {cnt}")

# The DataFrame equivalent (Catalyst-optimized, usually preferred in production)
text_df = spark.createDataFrame([(line,) for line in sample_text], ["text"])
print("\nSame result via DataFrame API:")
text_df.select(explode(split(lower(col("text")), " ")).alias("word")) \
    .groupBy("word").count().orderBy(desc("count")).show(10)


=== Classic MapReduce: Word Count ===
Top 10 words (RDD MapReduce):
  spark: 6
  processing: 3
  and: 3
  for: 2
  data: 2
  engine: 1
  python: 1
  r: 1
  hadoop: 1
  yarn: 1

Same result via DataFrame API:


+----------+-----+
|      word|count|
+----------+-----+
|     spark|    6|
|processing|    3|
|       and|    3|
|       for|    2|
|      data|    2|
|        is|    1|
|    engine|    1|
|         a|    1|
|   unified|    1|
| analytics|    1|
+----------+-----+
only showing top 10 rows


In [27]:
print("=== MapReduce on our e-commerce data: revenue by category ===")

def extract_purchase_info(json_str):
    try:
        event = json.loads(json_str)
        if event.get("event_type") == "purchase" and event.get("total_amount"):
            return [(event.get("category", "Unknown"), event["total_amount"])]
        return []
    except Exception:
        return []

raw_rdd = spark.sparkContext.parallelize(raw_json_events)
category_sales_rdd = raw_rdd.flatMap(extract_purchase_info)
total_by_category = category_sales_rdd.reduceByKey(lambda a, b: a + b)

print("Revenue by category (RDD MapReduce):")
for category, total in total_by_category.sortBy(lambda x: -x[1]).collect():
    print(f"  {category}: ${total:,.2f}")


=== MapReduce on our e-commerce data: revenue by category ===
Revenue by category (RDD MapReduce):
  Books: $103,687.69
  Clothing: $99,244.93
  Electronics: $98,662.25
  Beauty: $95,249.05
  Home: $90,974.91
  Sports: $84,735.31


## Part 9: RDD vs DataFrame - Performance Comparison

DataFrames go through Spark's **Catalyst optimizer** and **Tungsten** execution engine, which is why
they usually outperform hand-written RDD code for the same logical operation.


In [28]:
sample_data = events_df.limit(1500).collect()

print("--- RDD Approach ---")
start_time = time.time()
rdd = spark.sparkContext.parallelize(sample_data)
rdd_result = rdd.filter(lambda x: x["event_type"] == "purchase") \
    .map(lambda x: (x["device"], 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .collect()
rdd_time = time.time() - start_time
print(f"RDD result: {rdd_result} | time: {rdd_time:.4f}s")

print("\n--- DataFrame Approach ---")
start_time = time.time()
df_result = events_df.limit(1500) \
    .filter(col("event_type") == "purchase") \
    .groupBy("device").count().collect()
df_time = time.time() - start_time
print(f"DataFrame result: {[(r['device'], r['count']) for r in df_result]} | time: {df_time:.4f}s")

print("\nRecommendation: prefer DataFrames for most work; drop to RDDs only for custom logic")


--- RDD Approach ---
RDD result: [('mobile', 49), ('tablet', 75), ('desktop', 52)] | time: 0.2677s

--- DataFrame Approach ---
DataFrame result: [('mobile', 49), ('tablet', 75), ('desktop', 52)] | time: 0.3908s

Recommendation: prefer DataFrames for most work; drop to RDDs only for custom logic


## Part 10: The Medallion Architecture (Bronze -> Silver -> Gold)

```
+-------------------------------------------------------------------------+
|                      MEDALLION ARCHITECTURE                             |
+-------------------------------------------------------------------------+
|                                                                         |
|   RAW DATA        BRONZE LAYER       SILVER LAYER       GOLD LAYER     |
|                                                                         |
|   +---------+    +-------------+    +-------------+    +-------------+ |
|   |  JSON   |--->| Raw Events  |--->|  Cleaned &  |--->|  Business   | |
|   |  CSV    |    | + Metadata  |    |  Validated  |    |  Metrics    | |
|   |  APIs   |    | + Lineage   |    |  + Typed    |    |  + KPIs     | |
|   +---------+    +-------------+    +-------------+    +-------------+ |
|                                                                         |
+-------------------------------------------------------------------------+
```

- **Bronze**: store data exactly as received, plus ingestion metadata, for full lineage/replay.
- **Silver**: clean, validate, cast types, and quarantine bad records.
- **Gold**: business-ready aggregations that power dashboards and downstream analytics.

We persist Bronze and Silver to `~/spark-lab-data/shared/` so that **Lab 3 loads this exact Silver
dataset** instead of regenerating its own.


In [29]:
BRONZE_DIR = f"{SHARED_DIR}/bronze"
SILVER_DIR = f"{SHARED_DIR}/silver"
GOLD_DIR = f"{SHARED_DIR}/gold"

for directory in [BRONZE_DIR, SILVER_DIR, GOLD_DIR]:
    os.makedirs(directory, exist_ok=True)
    print(f"Created: {directory}")


Created: /Users/martin/spark-lab-data/shared/bronze
Created: /Users/martin/spark-lab-data/shared/silver
Created: /Users/martin/spark-lab-data/shared/gold


In [30]:
print("=" * 60)
print("BRONZE LAYER: Raw Data Ingestion")
print("=" * 60)

def parse_event_with_metadata(json_str):
    """Parse raw JSON and attach Bronze layer lineage metadata."""
    try:
        event = json.loads(json_str)
        event["_bronze_ingestion_time"] = datetime.now().isoformat()
        event["_bronze_source"] = "raw_clickstream_feed"
        event["_bronze_status"] = "valid"
        event["_bronze_raw_data"] = json_str
        return event
    except Exception as e:
        return {
            "_bronze_ingestion_time": datetime.now().isoformat(),
            "_bronze_source": "raw_clickstream_feed",
            "_bronze_status": "parse_error",
            "_bronze_error": str(e),
            "_bronze_raw_data": json_str,
        }

bronze_rdd = raw_rdd.map(parse_event_with_metadata)
bronze_df = spark.createDataFrame(bronze_rdd)

print(f"Total records: {bronze_df.count()}")
for row in bronze_df.groupBy("_bronze_status").count().collect():
    print(f"  {row['_bronze_status']}: {row['count']}")


BRONZE LAYER: Raw Data Ingestion


Total records: 6000


  valid: 6000


In [31]:
bronze_df.printSchema()
print("Sample Bronze records:")
bronze_df.filter(col("_bronze_status") == "valid") \
    .select("event_id", "timestamp", "user_id", "event_type", "_bronze_status") \
    .show(5, truncate=False)

bronze_path = f"{BRONZE_DIR}/events"
bronze_df.write.mode("overwrite").parquet(bronze_path)
print(f"Bronze layer saved to: {bronze_path}")


root
 |-- _bronze_ingestion_time: string (nullable = true)
 |-- _bronze_raw_data: string (nullable = true)
 |-- _bronze_source: string (nullable = true)
 |-- _bronze_status: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- price: double (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- search_query: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- user_id: long (nullable = true)

Sample Bronze records:
+----------+-------------------+-------+----------------+--------------+
|event_id  |timestamp          |user_id|event_type      |_bronze_status|
+----------+-------------------+-------+----------------+--------------+
|evt_000001|2024-01-21T03:01:47|63

Bronze layer saved to: /Users/martin/spark-lab-data/shared/bronze/events


### Silver Layer: Cleaning, Validation & Type Casting

In [32]:
print("=" * 60)
print("SILVER LAYER: Data Cleaning & Validation")
print("=" * 60)

bronze_df = spark.read.parquet(bronze_path)
valid_bronze = bronze_df.filter(col("_bronze_status") == "valid")
print(f"Starting with {valid_bronze.count()} valid Bronze records")

silver_df = valid_bronze \
    .withColumn("event_timestamp", expr("try_cast(timestamp AS timestamp)")) \
    .withColumn("event_date", to_date(col("event_timestamp"))) \
    .withColumn("event_hour", hour(col("event_timestamp"))) \
    .withColumn("day_of_week", dayofweek(col("event_date")))

silver_df = silver_df.withColumn(
    "user_id_clean",
    when(col("user_id").cast("int").isNotNull(), col("user_id").cast("int")).otherwise(lit(None))
)

silver_df = silver_df \
    .withColumn("device", lower(trim(col("device")))) \
    .withColumn("country", upper(trim(col("country")))) \
    .withColumn("event_type", lower(trim(col("event_type")))) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("quantity", col("quantity").cast("int")) \
    .withColumn("total_amount", col("total_amount").cast("double"))

silver_df = silver_df \
    .withColumn("_silver_processed_time", lit(datetime.now().isoformat())) \
    .withColumn("_silver_is_valid",
                when(
                    col("user_id_clean").isNotNull() &
                    col("event_timestamp").isNotNull() &
                    col("event_type").isNotNull(),
                    lit(True)
                ).otherwise(lit(False)))

print("Transformations applied")


SILVER LAYER: Data Cleaning & Validation
Starting with 6000 valid Bronze records
Transformations applied


In [33]:
total_records = silver_df.count()
valid_records = silver_df.filter(col("_silver_is_valid") == True).count()
invalid_records = total_records - valid_records

print("--- Silver Layer Data Quality Report ---")
print(f"Total: {total_records} | Valid: {valid_records} ({valid_records/total_records*100:.1f}%) "
      f"| Invalid: {invalid_records} ({invalid_records/total_records*100:.1f}%)")


--- Silver Layer Data Quality Report ---
Total: 6000 | Valid: 5809 (96.8%) | Invalid: 191 (3.2%)


In [34]:
silver_valid = silver_df.filter(col("_silver_is_valid") == True)
silver_quarantine = silver_df.filter(col("_silver_is_valid") == False)

silver_final = silver_valid.select(
    col("event_id"), col("event_timestamp"), col("event_date"), col("event_hour"),
    col("day_of_week"), col("user_id_clean").alias("user_id"), col("event_type"),
    col("device"), col("country"), col("session_id"), col("product_id"),
    col("category"), col("price"), col("quantity"), col("total_amount"),
    col("_silver_processed_time")
)

print(f"Silver Layer final: {silver_final.count()} valid records")
print(f"Quarantined: {silver_quarantine.count()} records")
silver_final.printSchema()

silver_path = f"{SILVER_DIR}/events"
quarantine_path = f"{SILVER_DIR}/quarantine"
silver_final.write.mode("overwrite").parquet(silver_path)
silver_quarantine.write.mode("overwrite").parquet(quarantine_path)
print(f"Silver layer saved to: {silver_path}")
print(f"Quarantine saved to: {quarantine_path}")


Silver Layer final: 5809 valid records
Quarantined: 191 records
root
 |-- event_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- device: string (nullable = true)
 |-- country: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- _silver_processed_time: string (nullable = false)

Silver layer saved to: /Users/martin/spark-lab-data/shared/silver/events
Quarantine saved to: /Users/martin/spark-lab-data/shared/silver/quarantine


In [35]:
silver_final.show(10, truncate=False)


+----------+-------------------+----------+----------+-----------+-------+----------------+-------+-------+------------+----------+--------+------+--------+------------+--------------------------+
|event_id  |event_timestamp    |event_date|event_hour|day_of_week|user_id|event_type      |device |country|session_id  |product_id|category|price |quantity|total_amount|_silver_processed_time    |
+----------+-------------------+----------+----------+-----------+-------+----------------+-------+-------+------------+----------+--------+------+--------+------------+--------------------------+
|evt_000001|2024-01-21 03:01:47|2024-01-21|3         |1          |63     |purchase        |mobile |DE     |sess_63_001 |prod_0087 |Beauty  |447.16|1       |447.16      |2026-10-02T11:39:10.171022|
|evt_000002|2024-01-02 00:05:13|2024-01-02|0         |3          |130    |remove_from_cart|tablet |US     |sess_130_005|NULL      |NULL    |NULL  |NULL    |NULL        |2026-10-02T11:39:10.171022|
|evt_000003|202

### Gold Layer: Business-Ready Aggregations

In [36]:
print("=" * 60)
print("GOLD LAYER: Business Aggregations")
print("=" * 60)

silver_df = spark.read.parquet(silver_path)
silver_df.cache()
print(f"Loaded {silver_df.count()} Silver records for Gold layer processing")


GOLD LAYER: Business Aggregations
Loaded 5809 Silver records for Gold layer processing


In [37]:
daily_metrics = silver_df.groupBy("event_date").agg(
    count("*").alias("total_events"),
    countDistinct("user_id").alias("unique_users"),
    countDistinct("session_id").alias("unique_sessions"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("num_purchases"),
    spark_sum(when(col("event_type") == "purchase", col("total_amount")).otherwise(0)).alias("daily_revenue"),
).withColumn(
    "conversion_rate", spark_round(col("num_purchases") / col("unique_users") * 100, 2)
).withColumn(
    "avg_order_value", spark_round(col("daily_revenue") / col("num_purchases"), 2)
).orderBy("event_date")

daily_metrics.show(10)


+----------+------------+------------+---------------+-------------+------------------+---------------+---------------+
|event_date|total_events|unique_users|unique_sessions|num_purchases|     daily_revenue|conversion_rate|avg_order_value|
+----------+------------+------------+---------------+-------------+------------------+---------------+---------------+
|2024-01-01|         203|         125|            178|           27|          23101.82|           21.6|         855.62|
|2024-01-02|         179|         120|            163|           24|           16660.7|           20.0|          694.2|
|2024-01-03|         170|         116|            164|           18|14356.960000000003|          15.52|         797.61|
|2024-01-04|         178|         112|            160|           24|          14966.07|          21.43|         623.59|
|2024-01-05|         199|         132|            178|           22|          20561.91|          16.67|         934.63|
|2024-01-06|         191|         123|  

In [38]:
user_metrics = silver_df.groupBy("user_id").agg(
    count("*").alias("total_events"),
    countDistinct("session_id").alias("total_sessions"),
    countDistinct("event_date").alias("active_days"),
    spark_min("event_timestamp").alias("first_seen"),
    spark_max("event_timestamp").alias("last_seen"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("num_purchases"),
    spark_sum(when(col("event_type") == "purchase", col("total_amount")).otherwise(0)).alias("total_spent"),
    first("device").alias("primary_device"),
).withColumn(
    "user_segment",
    when(col("total_spent") > 500, "High Value")
    .when(col("total_spent") > 100, "Medium Value")
    .when(col("total_spent") > 0, "Low Value")
    .otherwise("Non-Purchaser")
).orderBy(desc("total_spent"))

user_metrics.show(15)


+-------+------------+--------------+-----------+-------------------+-------------------+-------------+-----------------+--------------+------------+
|user_id|total_events|total_sessions|active_days|         first_seen|          last_seen|num_purchases|      total_spent|primary_device|user_segment|
+-------+------------+--------------+-----------+-------------------+-------------------+-------------+-----------------+--------------+------------+
|    178|          43|             5|         23|2024-01-01 09:44:12|2024-01-30 06:47:47|            7|          9000.78|       desktop|  High Value|
|     56|          37|             5|         23|2024-01-01 02:28:29|2024-01-30 16:09:14|            5|          8879.11|        tablet|  High Value|
|     51|          33|             5|         22|2024-01-01 15:59:00|2024-01-30 23:57:14|            6|8732.289999999999|        mobile|  High Value|
|     29|          38|             5|         22|2024-01-01 09:12:13|2024-01-30 21:23:56|           

In [39]:
product_metrics = silver_df.filter(col("product_id").isNotNull()).groupBy("product_id", "category").agg(
    count("*").alias("total_interactions"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("purchases"),
    spark_sum(when(col("event_type") == "purchase", col("total_amount")).otherwise(0)).alias("revenue"),
    countDistinct("user_id").alias("unique_users"),
).orderBy(desc("revenue"))

product_metrics.show(15)


+----------+-----------+------------------+---------+------------------+------------+
|product_id|   category|total_interactions|purchases|           revenue|unique_users|
+----------+-----------+------------------+---------+------------------+------------+
| prod_0035|     Beauty|                 7|        3|           6619.37|           7|
| prod_0084|      Books|                 5|        3|           5769.33|           5|
| prod_0049|     Beauty|                 6|        3|           5627.55|           6|
| prod_0095|Electronics|                10|        5|           5251.04|          10|
| prod_0056|     Beauty|                12|        4|           5010.71|          12|
| prod_0025|      Books|                 6|        4|           4669.59|           6|
| prod_0026|Electronics|                 6|        4|           4377.79|           6|
| prod_0001|   Clothing|                 9|        5|           4373.37|           9|
| prod_0036|     Beauty|                 6|        4| 

In [40]:
category_metrics = silver_df.filter(col("category").isNotNull()).groupBy("category").agg(
    count("*").alias("total_events"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("num_purchases"),
    spark_round(spark_sum(when(col("event_type") == "purchase", col("total_amount")).otherwise(0)), 2).alias("total_revenue"),
    countDistinct("user_id").alias("unique_customers"),
).orderBy(desc("total_revenue"))

category_metrics.show()

daily_metrics.write.mode("overwrite").parquet(f"{GOLD_DIR}/daily_metrics")
user_metrics.write.mode("overwrite").parquet(f"{GOLD_DIR}/user_metrics")
product_metrics.write.mode("overwrite").parquet(f"{GOLD_DIR}/product_metrics")
category_metrics.write.mode("overwrite").parquet(f"{GOLD_DIR}/category_metrics")
print("All Gold tables saved")

silver_df.unpersist()


+-----------+------------+-------------+-------------+----------------+
|   category|total_events|num_purchases|total_revenue|unique_customers|
+-----------+------------+-------------+-------------+----------------+
|      Books|         461|          116|    101955.68|             181|
|   Clothing|         475|          125|     98866.13|             182|
|Electronics|         480|          112|     94109.89|             181|
|     Beauty|         475|          108|     93842.97|             184|
|       Home|         509|          114|     90974.91|             186|
|     Sports|         435|          113|     83075.18|             180|
+-----------+------------+-------------+-------------+----------------+

All Gold tables saved


DataFrame[event_id: string, event_timestamp: timestamp, event_date: date, event_hour: int, day_of_week: int, user_id: int, event_type: string, device: string, country: string, session_id: string, product_id: string, category: string, price: double, quantity: int, total_amount: double, _silver_processed_time: string]

## Part 11: Pipeline Validation Summary

In [41]:
bronze_count = spark.read.parquet(bronze_path).count()
silver_count = spark.read.parquet(silver_path).count()
quarantine_count = spark.read.parquet(quarantine_path).count()

print("Data Flow Summary:")
print(f"  Raw events generated:   {len(raw_json_events):,}")
print(f"  Bronze layer records:   {bronze_count:,}")
print(f"  Silver layer records:   {silver_count:,}")
print(f"  Quarantined records:    {quarantine_count:,}")
print(f"  Data quality rate:      {silver_count/bronze_count*100:.1f}%")

print("\nGold layer tables:")
for table in ["daily_metrics", "user_metrics", "product_metrics", "category_metrics"]:
    cnt = spark.read.parquet(f"{GOLD_DIR}/{table}").count()
    print(f"  {table}: {cnt:,} rows")


Data Flow Summary:
  Raw events generated:   6,000
  Bronze layer records:   6,000
  Silver layer records:   5,809
  Quarantined records:    191
  Data quality rate:      96.8%

Gold layer tables:
  daily_metrics: 30 rows
  user_metrics: 200 rows
  product_metrics: 593 rows
  category_metrics: 6 rows


## Lab 2 Summary

**Key concepts learned**

- Spark architecture: driver, executors, tasks, partitions
- Converting between RDD, DataFrame, and SQL views
- Column operations, type casting, and null handling
- Filtering, sorting, and aggregations
- Reading/writing Parquet, CSV, JSON, and partitioned data
- MapReduce patterns with RDDs (`map`, `flatMap`, `reduceByKey`)
- The Medallion Architecture: Bronze (raw + lineage) -> Silver (cleaned + quarantine) -> Gold (KPIs)

**Data saved for Lab 3** (`~/spark-lab-data/shared/`):

- `bronze/events` - raw parsed events with lineage metadata
- `silver/events` and `silver/quarantine` - cleaned, typed, validated data
- `gold/daily_metrics`, `gold/user_metrics`, `gold/product_metrics`, `gold/category_metrics`

Lab 3 loads the **Silver layer** produced here and focuses on window functions, partitioning,
caching, joins, UDFs, structured streaming, and production patterns.

In [42]:
spark.stop()
print("Lab 2 complete! Continue with Lab 3: Advanced Spark & Production Patterns")


Lab 2 complete! Continue with Lab 3: Advanced Spark & Production Patterns
